In [9]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np
import os
from tqdm.notebook import tnrange, tqdm
import glob
# import math
import scipy
# import deeplabcut
import re
import biteOscope_utils

In [10]:
experiment = '/mnt/DATA2/zhong/BiteOScope_Data/20250418_An_Stephen_vivax/Individual_Tracklets/NotEngorged'
saveDir =  '/mnt/DATA2/zhong/BiteOScope_Data/Result_Stat'
min_length = 25
hdfs = glob.glob(f'{experiment}/*.h5')

In [ ]:
minLen = min_length     ### minimum length of a trajectory to be included
smoothLenXY = 3         ### averaging window to create smoothed x and y
smoothLenDist = 5       ### averaging window to create smoothed distance
smoothLenBelly = 25     ### averaging window to create smoothed belly width
minEngInc = 1.2         ### minimum increase of abdominal width for mosquito to be scored as engorged
minEngW = 18            ### minimal abdominal width for a mosquito to be scored as engorged
minEngTime = 400        ### minimal time (in frames) in which a mosquito can engorge
maxMinW = 18.5          ### minimal abdominal width to be scored as 'already engorged'
extraLenEngorge = 20000 ### tracking data is truncated this number of frames after a mosquito is scored as engorged
movingThres = 2.5       ### minimum distance to cover between two frames to be scored as 'moving'
pCutOff = 0.8           
probeFilterWindow = 3
hardMinBelly = 7        #### Outliers      
hardMaxBelly = 42       #### Outliers  
minNonNan = 10          ####


In [ ]:
def trackStats_engorge1(experiment, saveDir):
    hdfs = glob.glob(f'{experiment}/*.h5')
    indexes = []
    trackData = pd.DataFrame(columns=['experiment', 'infected', 'ID', 'totDist', 'totDistSm', 'totTime', 'meanSpeed',
                                        'movingTime', 'movingFrac', 'bellyInc', 'bellyMax09', 'bellyMin01', 'engorged', 
                                      'timeToEngorge', 'startingFrame','date','hdf'], 
                            index=indexes)
    for hdf in tqdm(hdfs):
        Dg = pd.read_hdf(hdf)
        Dg_name = os.path.splitext(os.path.basename(hdf))[0][57:62]
        scorer = Dg.columns.get_level_values(0).unique()[0]
        ind = Dg.columns.get_level_values(1).unique()[0]
        experiment = os.path.splitext(os.path.basename(hdf))[0][0:3]
        ID = experiment + '_' + os.path.splitext(os.path.basename(hdf))[0][-5:]
        Date = os.path.splitext(os.path.basename(hdf))[0][4:12]
        if len(Dg) > minLen:
            for bp in Dg.columns.get_level_values(2).unique():
                Dg[scorer, ind, bp, 'x'].mask(Dg[scorer, ind, bp, 'likelihood'] < pCutOff,inplace=True)
                Dg[scorer, ind, bp, 'y'].mask(Dg[scorer, ind, bp, 'likelihood'] < pCutOff,inplace=True)
                Dg[scorer, ind, bp, 'xSm'] = Dg[scorer, ind, bp, 'x'].interpolate(limit = 2)
                Dg[scorer, ind, bp, 'xSm'] = Dg[scorer, ind, bp, 'xSm'].rolling(smoothLenXY, min_periods=1, center=True).mean()
                Dg[scorer, ind, bp, 'ySm'] = Dg[scorer, ind, bp, 'y'].interpolate(limit = 2)
                Dg[scorer, ind, bp, 'ySm'] = Dg[scorer, ind, bp, 'ySm'].rolling(smoothLenXY, min_periods=1, center=True).mean()
                Dg[scorer, ind, bp, 'distance'] = \
                np.linalg.norm([Dg[scorer][ind][bp].xSm.diff().astype(float),
                                Dg[scorer][ind][bp].ySm.diff().astype(float)], axis=0)
                Dg.replace([np.inf, -np.inf], np.nan, inplace=True)
                Dg[scorer, ind, bp, 'distance'].interpolate(inplace = True)
                if len(Dg) > smoothLenDist:
                    Dg[scorer, ind, bp, 'distanceSm'] = Dg[scorer, ind, bp, 'distance'].rolling(smoothLenDist, min_periods=1, center=True).mean()
                else:
                    Dg[scorer, ind, bp, 'distanceSm'] = Dg[scorer, ind, bp, 'distance']
            Dg[scorer, ind, 'abdomenC', 'bellyW'] = \
            np.linalg.norm([Dg[scorer][ind]['abdomenL'].x.astype(float) - Dg[scorer][ind]['abdomenR'].x.astype(float),
                            Dg[scorer][ind]['abdomenL'].y.astype(float) - Dg[scorer][ind]['abdomenR'].y.astype(float)], axis=0)
            if len(Dg) > smoothLenBelly:
                Dg[scorer, ind, 'abdomenC', 'bellyWSm'] = \
                np.linalg.norm([Dg[scorer][ind]['abdomenL'].x.rolling(smoothLenBelly, min_periods=1).mean() -
                                Dg[scorer][ind]['abdomenR'].x.rolling(smoothLenBelly, min_periods=1).mean(),
                                Dg[scorer][ind]['abdomenL'].y.rolling(smoothLenBelly, min_periods=1).mean() -
                                Dg[scorer][ind]['abdomenR'].y.rolling(smoothLenBelly, min_periods=1).mean()],
                               axis=0)
                Dg[scorer, ind, 'abdomenC', 'bellyWSm'] = Dg[scorer, ind, 'abdomenC', 'bellyWSm'].rolling(smoothLenBelly, min_periods=1).mean()
                Dg[scorer, ind, 'abdomenC', 'bellyWSm'].mask(Dg[scorer, ind, 'abdomenC', 'bellyWSm'] < hardMinBelly, inplace=True)
                Dg[scorer, ind, 'abdomenC', 'bellyWSm'].mask(Dg[scorer, ind, 'abdomenC', 'bellyWSm'] > hardMaxBelly, inplace=True)
            else:
                Dg[scorer, ind, 'abdomenC', 'bellyWSm'] = Dg[scorer, ind, 'abdomenC', 'bellyW']
                Dg[scorer, ind, 'abdomenC', 'bellyWSm'].mask(Dg[scorer, ind, 'abdomenC', 'bellyWSm'] < hardMinBelly, inplace=True)
                Dg[scorer, ind, 'abdomenC', 'bellyWSm'].mask(Dg[scorer, ind, 'abdomenC', 'bellyWSm'] > hardMaxBelly, inplace=True)

            Dg[scorer, ind, 'abdomenC', 'engorged'] = 0

            if len(Dg.loc[Dg[scorer][ind]['abdomenC'].bellyW > 0]) > minEngTime:
                startI = Dg[scorer][ind]['abdomenC'].bellyWSm.first_valid_index()
                if ~np.isnan(startI):
                    bellyMax08 = Dg[scorer][ind]['abdomenC'].bellyWSm.quantile(.8)
                    bellyMax09 = Dg[scorer][ind]['abdomenC'].bellyWSm.quantile(.9)
                    bellyMin01 = Dg[scorer][ind]['abdomenC'].bellyWSm.loc[startI : startI + minEngTime].quantile(.1)
                    bellyInc = bellyMax09 / bellyMin01
                    if bellyMax09 > minEngW and bellyMin01 < maxMinW and bellyInc > minEngInc:
                        Dg[scorer, ind, 'abdomenC', 'engorged'] = Dg[scorer][ind]['abdomenC'].bellyWSm.where(Dg[scorer][ind]['abdomenC'].bellyWSm > bellyMax08) > 0
                        Dg[scorer, ind, 'abdomenC', 'engorged'] = Dg[scorer, ind, 'abdomenC', 'engorged'].astype(int)
                        timeToEngorge = Dg[scorer][ind]['abdomenC'].bellyWSm.where(Dg[scorer][ind]['abdomenC'].bellyWSm > bellyMax09 - 1).first_valid_index() - startI
                        engorgeFrameNo = Dg[scorer][ind]['abdomenC'].bellyWSm.where(Dg[scorer][ind]['abdomenC'].bellyWSm > bellyMax09 - 1).first_valid_index()
                        # Dg = Dg.truncate(after = engorgeFrameNo + extraLenEngorge)
                    elif bellyMax09 > minEngW and bellyMin01 > maxMinW:
                        Dg[scorer, ind, 'abdomenC', 'engorged'] = 2
                        timeToEngorge = Dg[scorer][ind]['abdomenC'].bellyWSm.where(Dg[scorer][ind]['abdomenC'].bellyWSm > bellyMax09 - 1).first_valid_index() - startI
                    else:
                        timeToEngorge = np.nan
            else:
                bellyMax08 = Dg[scorer][ind]['abdomenC'].bellyWSm.quantile(.8)
                bellyMax09 = Dg[scorer][ind]['abdomenC'].bellyWSm.quantile(.9)
                bellyMin01 = Dg[scorer][ind]['abdomenC'].bellyWSm.quantile(.1)
                bellyInc = bellyMax09 / bellyMin01
                timeToEngorge = np.nan

            movingTime = len(Dg[scorer, ind, 'thorax'].loc[Dg[scorer, ind, 'thorax', 'distanceSm'] > movingThres])    
            trackData.loc[ID, 'experiment'] = experiment
            if 'PF' in experiment or 'I' in experiment:
                trackData.loc[ID, 'infected'] = 1
            elif 'C' in experiment:
                trackData.loc[ID, 'infected'] = 0
            trackData.loc[ID, 'ID'] = ID
            distThorax = Dg[scorer, ind, 'thorax', 'distance'].sum()
            distHead = Dg[scorer, ind, 'head', 'distance'].sum()
            distAbdomen = Dg[scorer, ind, 'abdomenC', 'distance'].sum()
            trackData.loc[ID, 'totDist'] = np.nanmean([distThorax, distHead, distAbdomen])
            distThoraxSm = Dg[scorer, ind, 'thorax', 'distanceSm'].sum()
            distHeadSm = Dg[scorer, ind, 'head', 'distanceSm'].sum()
            distAbdomenSm = Dg[scorer, ind, 'abdomenC', 'distanceSm'].sum()
            # c = Dg[scorer, ind, 'stylet'].loc[Dg[scorer, ind, 'stylet', 'probe'] == 1].dropna().copy()
            # if len(c) > 0:
            #     numProbeLoc = np.sum([np.linalg.norm([c.x.diff().astype(float), c.y.diff().astype(float)], axis=0) > minDistProbeLoc])
            #     numProbeLoc += 1
            # else:
            #     numProbeLoc = 0                
            trackData.loc[ID, 'totDistSm'] = np.nanmean([distThoraxSm, distHeadSm, distAbdomenSm])
            trackData.loc[ID, 'totTime'] = Dg.last_valid_index() - Dg.first_valid_index() + 1
            trackData.loc[ID, 'bellyMax09'] = bellyMax09
            trackData.loc[ID, 'bellyMin01'] = bellyMin01
            trackData.loc[ID, 'bellyInc'] = bellyInc
            trackData.loc[ID, 'engorged'] = Dg[scorer, ind, 'abdomenC', 'engorged'].max()
            trackData.loc[ID, 'timeToEngorge'] = timeToEngorge
            trackData.loc[ID, 'movingTime'] = movingTime
            trackData.loc[ID, 'movingFrac'] = movingTime / len(Dg)
            trackData.loc[ID, 'startingFrame'] = Dg.first_valid_index()
            trackData.loc[ID, 'date'] = Date
            # trackData.loc[ID, 'probeDuration'] = Dg[scorer, ind, 'stylet', 'probe'].sum()
            # trackData.loc[ID, 'numProbes'] = np.sum(np.diff(Dg[scorer, ind, 'stylet', 'probe']) == 1).astype(int)        
            # trackData.loc[ID, 'numProbeLoc'] = numProbeLoc
            # probeStart = Dg[scorer, ind, 'stylet'].loc[Dg[scorer, ind, 'stylet'].probe == 1].first_valid_index()
            # probeStop = Ddf[scorer, 'ind1', 'head'].xg[scorer, ind, 'styallTracks'
            trackData.loc[ID, 'hdf'] = Dg_name

    trackData['meanSpeed'] = trackData['totDist'] / trackData['totTime']
    cols = ['infected', 'totDist', 'totDistSm', 'totTime', 'meanSpeed',
            'movingTime', 'movingFrac', 'bellyInc', 'bellyMax09', 'bellyMin01', 'engorged', 
            'timeToEngorge', 'startingFrame']
    for c in cols:
        trackData[c] = pd.to_numeric(trackData[c])
    output_name = os.path.join(saveDir, '/your/name.h5')
    trackData.to_hdf(output_name, "df_with_missing", format="table", mode="w")


In [ ]:
trackStats_engorge1(experiment, saveDir)

convert = pd.read_hdf('/your/name.h5')
convert.to_excel('/your/name.xlsx')

  0%|          | 0/3840 [00:00<?, ?it/s]